# Parallelization

This pattern enables additional efficiencies by running multiple sub-tasks in parallel and then collecting results for synthesis. Use cases include: deep research, travel planning, customer feedback analysis, etc.

## Implementation with Flyte v2

This notebook refactors the LangChain `RunnableParallel` example to use Flyte v2 and the native OpenAI client. The three parallel sub-tasks (summarize, questions, key terms) run as Flyte tasks that can execute in parallel containers when run remotely.

#### LangChain vs Flyte v2 — Key Differences

In the context of this pattern, these are the main differences between LangChain and Flyte V2:

| Aspect | LangChain | Flyte v2 |
|--------|-----------|----------|
| **Parallel execution** | `RunnableParallel` (LCEL) | `asyncio.gather` + Flyte tasks |
| **LLM calls** | `ChatOpenAI` via LCEL | Direct `AsyncAnthropic` client |
| **Caching** | None | `cache="auto"` per task |
| **Execution** | In-process only | Local or remote (containers) |
| **Dependencies** | langchain-openai, langchain-core | flyte, anthropic |

1. Install dependencies

In [ ]:
!uv pip install flyte anthropic

Using Python 3.12.12 environment at: /Users/davidmirror/code/agentic-patterns-on-v2/.venv
Audited 2 packages in 3ms


### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

2. Export your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...


3. Import dependencies and configure the Flyte TaskEnvironment

In [1]:
import os
import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret

flyte.init_from_config()

env = TaskEnvironment(
    name="parallel_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    image=flyte.Image.from_debian_base().with_pip_packages("anthropic>=0.25.0"),
    secrets=[Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
)


4. Define the three parallel sub-tasks and the synthesis task

Each sub-task is a Flyte task that calls Anthropic directly. When run remotely, Flyte schedules them in parallel containers.

In [2]:
async def _llm_call(system: str, user: str) -> str:
    """Helper: single LLM call with system + user messages."""
    from anthropic import AsyncAnthropic
    client = AsyncAnthropic()
    resp = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return resp.content[0].text.strip()


@env.task
async def summarize_task(topic: str) -> str:
    """Summarize the topic concisely."""
    return await _llm_call(
        "Summarize the following topic concisely:",
        topic,
    )


@env.task
async def questions_task(topic: str) -> str:
    """Generates questions about the topic."""
    return await _llm_call(
        "Generate three interesting questions about the following topic:",
        topic,
    )


@env.task
async def terms_task(topic: str) -> str:
    """Identify 5-10 key terms from the topic, comma-separated."""
    return await _llm_call(
        "Identify 5-10 key terms from the following topic, separated by commas:",
        topic,
    )


@env.task
async def synthesize_task(
    topic: str, summary: str, questions: str, key_terms: str
) -> str:
    """Synthesize a comprehensive answer from the parallel results."""
    system = """Based on the following information:
Summary: {summary}
Related Questions: {questions}
Key Terms: {key_terms}
Synthesize a comprehensive answer."""
    return await _llm_call(
        system.format(summary=summary, questions=questions, key_terms=key_terms),
        f"Original topic: {topic}",
    )


5. Orchestrate the parallel workflow

The main task runs the three sub-tasks in parallel with `asyncio.gather`. Flyte schedules them concurrently (locally or in separate containers when remote).

In [3]:
@env.task
async def parallel_research(topic: str) -> str:
    """Run summarize, questions, and terms in parallel, then synthesize."""
    summary, questions, key_terms = await asyncio.gather(
        summarize_task(topic),
        questions_task(topic),
        terms_task(topic),
    )
    return await synthesize_task(topic, summary, questions, key_terms)

6. Run in the devbox

In [4]:
run = flyte.run(parallel_research, topic="The history of space exploration")
run.wait()
print(run.outputs()[0])


> Building 1 image...

> Building image flyte for environment parallel_env

i Image localhost:30000/flyte:b7e21ad4642010db89f88189de935c8b already exists, skipping build

✓ Built image for environment parallel_env: localhost:30000/flyte:b7e21ad4642010db89f88189de935c8b

Output()

# A Comprehensive Overview of Space Exploration History

## The Foundational Narrative

Space exploration represents one of humanity's most remarkable achievements—a journey that transformed from Cold War competition into collaborative international effort. The field spans just over six decades yet encompasses milestones that fundamentally altered our understanding of the universe and our place within it.

## Historical Progression and Key Transitions

### **The Competitive Genesis (1957-1969)**
The Space Age began dramatically when the Soviet Union launched Sputnik 1 in 1957, shocking the Western world and triggering an intense technological rivalry. This competition, framed within Cold War tensions, accelerated innovation at an unprecedented pace. The Soviets achieved early dominance: Yuri Gagarin's 1961 orbital flight made him the first human in space. However, the United States' Apollo program, culminating in Neil Armstrong's 1969 Moon landing, marked a pivotal psychological and te

## Scaling the pattern

a. Use `ReusePolicy` to keep a pool of warm containers instead of starting a new one per action. When you have multiple short tasks (LLM calls) where container startup time dominates overall latency, Flyte's reusable containers eliminate that factor by enabling the action to run in an already-warm container. [Learn more](https://www.union.ai/docs/v2/union/user-guide/task-configuration/reusable-containers/)

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
reusable_image = flyte.Image.from_debian_base().with_pip_packages("unionai-reuse>=0.1.10")

env = flyte.TaskEnvironment(
    name="reusable-env",
    resources=flyte.Resources(memory="1Gi", cpu="500m"),
    reusable=flyte.ReusePolicy(
        replicas=2,                           # Create 2 container instances
        concurrency=1,                        # Process 1 task per container at a time
        scaledown_ttl=timedelta(minutes=10),  # Individual containers shut down after 5 minutes of inactivity
        idle_ttl=timedelta(hours=1)           # Entire environment shuts down after 30 minutes of no tasks
    ),
    image=reusable_image  # Use the container image augmented with the unionai-reuse library.
)

## Why not the Agent harness here?

Parallelization fans one input out to independent workers and joins the results. The control flow is **static and known ahead of time**, so `asyncio.gather` (or `flyte.map`) over `@env.task`s expresses it directly — and Flyte schedules the branches concurrently for free. An `Agent`'s tool-loop would insert an extra LLM round-trip just to decide what to run, which is pure overhead when the fan-out is fixed.